# RAG Experiment Analysis
LlamaIndex vs LangChain × Similarity Methods

In [ ]:
import sys, json, glob
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

plt.rcParams['font.family'] = 'AppleGothic'
sns.set_theme(style='whitegrid', font_scale=1.2)

In [ ]:
# Load the latest RAG result file
result_files = sorted(glob.glob(str(ROOT / 'results/raw/rag_results_*.json')))
if not result_files:
    print('No result files found. Run experiments/run_experiments.py first.')
else:
    latest = result_files[-1]
    print(f'Loading: {latest}')
    with open(latest) as f:
        raw = json.load(f)
    df = pd.DataFrame(raw)
    print(f'Rows: {len(df)}, Columns: {list(df.columns)}')

In [ ]:
# ── Summary table
summary = df.groupby(['framework', 'method']).agg(
    total_time_s=('total_time_s', 'mean'),
    retrieval_time_s=('retrieval_time_s', 'mean'),
    gen_time_s=('gen_time_s', 'mean'),
    avg_ctx_relevance=('avg_ctx_relevance', 'mean'),
    answer_length=('answer_length', 'mean'),
).round(4)
summary

In [ ]:
# ── Context relevance per method (both frameworks)
pivot = df.pivot_table(index='method', columns='framework', values='avg_ctx_relevance', aggfunc='mean')
pivot.plot(kind='bar', figsize=(10, 5), colormap='Set1', edgecolor='white')
plt.title('Average Context Relevance by Similarity Method', fontweight='bold')
plt.ylabel('Cosine Similarity (query ↔ retrieved chunks)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(ROOT / 'results/plots/nb_ctx_relevance.png', dpi=150)
plt.show()

In [ ]:
# ── Latency breakdown
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, fw in zip(axes, ['llamaindex', 'langchain']):
    sub = df[df['framework'] == fw].groupby('method')[['retrieval_time_s', 'gen_time_s']].mean().sort_values('retrieval_time_s')
    sub.plot(kind='bar', stacked=True, ax=ax, colormap='Set2', edgecolor='white')
    ax.set_title(f'{fw.upper()} – Latency Breakdown', fontweight='bold')
    ax.set_ylabel('Seconds')
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig(ROOT / 'results/plots/nb_latency.png', dpi=150)
plt.show()

In [ ]:
# ── Per-query deep dive
query_idx = 0   # change to inspect a different query
q = df['question'].unique()[query_idx]
print(f'Query: {q}\n')

sub = df[df['question'] == q][['framework', 'method', 'answer', 'avg_ctx_relevance', 'total_time_s']]
for _, row in sub.iterrows():
    print(f"[{row['framework']:12s} / {row['method']:12s}]  ctx_rel={row['avg_ctx_relevance']:.3f}  t={row['total_time_s']:.2f}s")
    print(f"  Answer: {row['answer'][:200]}...\n")

In [ ]:
# ── Heatmap: context relevance
pivot2 = df.pivot_table(index='method', columns='framework', values='avg_ctx_relevance', aggfunc='mean')
plt.figure(figsize=(7, 5))
sns.heatmap(pivot2, annot=True, fmt='.3f', cmap='YlOrRd', linewidths=0.5)
plt.title('Context Relevance Heatmap', fontweight='bold')
plt.tight_layout()
plt.savefig(ROOT / 'results/plots/nb_heatmap.png', dpi=150)
plt.show()

In [ ]:
# ── Similarity-only benchmark (no LLM)
sim_path = ROOT / 'results/raw/similarity_only.json'
if sim_path.exists():
    sim_df = pd.DataFrame(json.loads(sim_path.read_text()))
    print(sim_df.groupby('method')[['elapsed_s', 'top1_score']].mean().round(5))
else:
    print('Run with --similarity-only flag to generate this file.')